# Correlation Feasibility Analysis for CuBench Block C (Macro) Features

This notebook empirically tests the feature relationships assumed by this project's
literature-review documents (`paper1/literature/legacy_review/literature_review_copper_price_forecasting.md`,
`paper1/literature/legacy_review/copper_fundamentals.md`) against **this project's own real market data pull**.

Those documents cite *other papers'* findings on *other* datasets/periods (e.g. "DXY and
copper exhibit a well-documented negative correlation, typically -0.3 to -0.6"; gold/silver
have "great/high influence"; China PMI is "positively correlated, particularly at 1-3 month
horizons"; VIX is "negatively correlated"). None of these claims have been checked against
data actually pulled for this project. This notebook does that check.

Data: daily closes 2010-01-01 -> today via `yfinance`, plus FRED free CSV endpoint series.

In [1]:
import json
import time
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import yfinance as yf
from scipy import stats
from statsmodels.tsa.stattools import adfuller, coint, grangercausalitytests

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

REPO_ROOT = Path(r"D:\copper\paper2")  # updated post repo-restructure: correlation data/results now live under paper2/
RAW_DIR = REPO_ROOT / "data" / "raw_correlation_check"
RESULTS_DIR = REPO_ROOT / "results" / "correlation_analysis"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2010-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")
print(f"Pulling data from {START_DATE} to {END_DATE}")

Pulling data from 2010-01-01 to 2026-08-19


## 1. Data acquisition

### 1a. Yahoo Finance tickers (via `yfinance`), with retry/backoff and local CSV caching.

In [2]:
YF_TICKERS = {
    "copper": "HG=F",
    "aluminum": "ALI=F",
    "gold": "GC=F",
    "silver": "SI=F",
    "crude_oil": "CL=F",
    "dxy": "DX-Y.NYB",
    "sp500": "^GSPC",
    "vix": "^VIX",
    "us10y_yield": "^TNX",
}


def download_yf_with_retry(ticker, start, end, max_retries=5, base_delay=5):
    for attempt in range(1, max_retries + 1):
        try:
            df = yf.download(
                ticker, start=start, end=end, auto_adjust=False,
                progress=False, threads=False,
            )
            if df is None or df.empty:
                raise ValueError(f"Empty dataframe returned for {ticker}")
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            return df
        except Exception as e:
            wait = base_delay * (2 ** (attempt - 1))
            print(f"  [{ticker}] attempt {attempt}/{max_retries} failed: {e}. "
                  f"{'Retrying in ' + str(wait) + 's' if attempt < max_retries else 'Giving up.'}")
            if attempt < max_retries:
                time.sleep(wait)
    return None


yf_status = {}
yf_data = {}
for name, ticker in YF_TICKERS.items():
    cache_path = RAW_DIR / f"yf_{name}.csv"
    if cache_path.exists():
        print(f"[{name}] loading from cache {cache_path.name}")
        df = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        yf_data[name] = df
        yf_status[name] = "loaded_from_cache"
        continue
    print(f"[{name}] downloading {ticker} from Yahoo Finance...")
    df = download_yf_with_retry(ticker, START_DATE, END_DATE)
    if df is not None:
        df.to_csv(cache_path)
        yf_data[name] = df
        yf_status[name] = "downloaded_ok"
        print(f"  -> {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")
    else:
        yf_status[name] = "FAILED"
        print(f"  -> FAILED to download {ticker} after retries.")
    time.sleep(1.5)  # be polite between tickers

print("\nYahoo Finance download status:")
for k, v in yf_status.items():
    print(f"  {k}: {v}")

[copper] loading from cache yf_copper.csv
[aluminum] loading from cache yf_aluminum.csv
[gold] loading from cache yf_gold.csv
[silver] loading from cache yf_silver.csv
[crude_oil] loading from cache yf_crude_oil.csv
[dxy] loading from cache yf_dxy.csv
[sp500] loading from cache yf_sp500.csv
[vix] loading from cache yf_vix.csv
[us10y_yield] loading from cache yf_us10y_yield.csv

Yahoo Finance download status:
  copper: loaded_from_cache
  aluminum: loaded_from_cache
  gold: loaded_from_cache
  silver: loaded_from_cache
  crude_oil: loaded_from_cache
  dxy: loaded_from_cache
  sp500: loaded_from_cache
  vix: loaded_from_cache
  us10y_yield: loaded_from_cache


### 1b. FRED free no-API-key CSV endpoint

`https://fred.stlouisfed.org/graph/fredgraph.csv?id=SERIES_ID`

- `DFII10`: 10-Year Treasury Inflation-Indexed Security, real yield (daily)
- `BAMLH0A0HYM2`: ICE BofA US High Yield Index Option-Adjusted Spread (daily)
- China Manufacturing PMI: no free daily/monthly series with an official PMI reading was
  found on FRED under a no-API-key CSV endpoint. The genuine Caixin/S&P Global and NBS
  China Manufacturing PMI series are subscription-gated (S&P Global Market Intelligence)
  or published only as press-release numbers with no free bulk-download API.
  As a fallback, we pull FRED series `BSCICP03CNM665S` = OECD "Business Tendency Surveys
  (Manufacturing): Confidence Indicators: Composite Indicators: OECD Indicator for China",
  monthly, back to 2000 -- a widely used *free proxy* for Chinese manufacturing sentiment,
  but it is NOT the actual Caixin/NBS PMI series the literature review cites, and this
  distinction is preserved throughout the analysis and final verdict.

In [3]:
FRED_SERIES = {
    "real_yield_10y": "DFII10",
    "high_yield_oas": "BAMLH0A0HYM2",
    "china_pmi_proxy_oecd_business_confidence": "BSCICP03CNM665S",
}

FRED_NOTES = {
    "china_pmi_proxy_oecd_business_confidence": (
        "NOT the actual Caixin/NBS China Manufacturing PMI (those require paid "
        "subscription / have no free bulk CSV API). This is an OECD business-confidence "
        "proxy series (BSCICP03CNM665S), monthly, used as the best freely obtainable stand-in."
    )
}


def download_fred_with_retry(series_id, max_retries=5, base_delay=5):
    # NOTE: the `requests` library reliably times out against fred.stlouisfed.org on this
    # machine (schannel TLS renegotiation issue observed with curl -v as well, but curl
    # itself succeeds where requests/urllib3 does not). We shell out to curl instead, which
    # was empirically verified to work for this endpoint.
    import subprocess
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    for attempt in range(1, max_retries + 1):
        try:
            proc = subprocess.run(
                ["curl", "-s", "-m", "30", "-w", "\n__HTTP_CODE__%{http_code}", url],
                capture_output=True, text=True, timeout=45,
            )
            out = proc.stdout
            if "__HTTP_CODE__" not in out:
                raise ValueError(f"curl produced no status marker; stderr={proc.stderr[:200]}")
            body, code = out.rsplit("__HTTP_CODE__", 1)
            code = code.strip()
            if code != "200":
                raise ValueError(f"HTTP {code}")
            from io import StringIO
            df = pd.read_csv(StringIO(body))
            if df.empty or df.shape[1] < 2:
                raise ValueError("empty/malformed FRED CSV")
            date_col, val_col = df.columns[0], df.columns[1]
            df[date_col] = pd.to_datetime(df[date_col])
            df = df.set_index(date_col)
            df[val_col] = pd.to_numeric(df[val_col], errors="coerce")
            return df
        except Exception as e:
            wait = base_delay * (2 ** (attempt - 1))
            print(f"  [{series_id}] attempt {attempt}/{max_retries} failed: {e}. "
                  f"{'Retrying in ' + str(wait) + 's' if attempt < max_retries else 'Giving up.'}")
            if attempt < max_retries:
                time.sleep(wait)
    return None


fred_status = {}
fred_data = {}
for name, series_id in FRED_SERIES.items():
    cache_path = RAW_DIR / f"fred_{name}.csv"
    if cache_path.exists():
        print(f"[{name}] loading from cache {cache_path.name}")
        df = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        fred_data[name] = df
        fred_status[name] = "loaded_from_cache"
        continue
    print(f"[{name}] downloading FRED series {series_id}...")
    df = download_fred_with_retry(series_id)
    if df is not None:
        df.to_csv(cache_path)
        fred_data[name] = df
        fred_status[name] = "downloaded_ok"
        print(f"  -> {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")
    else:
        fred_status[name] = "FAILED"
    time.sleep(1.0)

print("\nFRED download status:")
for k, v in fred_status.items():
    print(f"  {k}: {v}")
if "high_yield_oas" in fred_data:
    hy = fred_data["high_yield_oas"]
    print(f"\nNOTE: BAMLH0A0HYM2 as served by the fredgraph.csv endpoint only returns "
          f"{hy.index.min().date()} to {hy.index.max().date()} ({len(hy)} rows) even though "
          f"the underlying ICE BofA HY OAS series exists back to 1996-12-31 on FRED's own "
          f"site. Requests with explicit cosd=2010-01-01 start-date parameters were tried and "
          f"made no difference (server-side default window on this endpoint). This is a real "
          f"data-access limitation of the free CSV endpoint, not a bug in this notebook -- "
          f"flagged here rather than silently working around it.")

[real_yield_10y] loading from cache fred_real_yield_10y.csv
[high_yield_oas] loading from cache fred_high_yield_oas.csv
[china_pmi_proxy_oecd_business_confidence] loading from cache fred_china_pmi_proxy_oecd_business_confidence.csv

FRED download status:
  real_yield_10y: loaded_from_cache
  high_yield_oas: loaded_from_cache
  china_pmi_proxy_oecd_business_confidence: loaded_from_cache

NOTE: BAMLH0A0HYM2 as served by the fredgraph.csv endpoint only returns 2023-08-21 to 2026-08-17 (793 rows) even though the underlying ICE BofA HY OAS series exists back to 1996-12-31 on FRED's own site. Requests with explicit cosd=2010-01-01 start-date parameters were tried and made no difference (server-side default window on this endpoint). This is a real data-access limitation of the free CSV endpoint, not a bug in this notebook -- flagged here rather than silently working around it.


China PMI: explicitly recorded as **NOT OBTAINABLE FREE in its genuine form** (Caixin/NBS).
We proceed with the OECD business-confidence proxy noted above, clearly labelled throughout.

In [4]:
print("China Manufacturing PMI (genuine Caixin/NBS series): NOT OBTAINABLE FREE.")
print("Reason: Caixin PMI is licensed via S&P Global Market Intelligence (subscription).")
print("        NBS official PMI is published as press-release numbers on data.stats.gov.cn")
print("        with no free bulk CSV/API for historical daily or monthly series.")
print("Fallback used: FRED BSCICP03CNM665S (OECD business confidence indicator, China,")
print("        manufacturing) -- a proxy only, not the literature's actual PMI series.")

data_source_log = {
    "yfinance": yf_status,
    "fred": fred_status,
    "china_pmi_genuine": "NOT_OBTAINABLE_FREE",
    "china_pmi_fallback": "FRED BSCICP03CNM665S (OECD business confidence proxy, monthly)",
}
with open(RAW_DIR / "data_source_log.json", "w") as f:
    json.dump(data_source_log, f, indent=2, default=str)

China Manufacturing PMI (genuine Caixin/NBS series): NOT OBTAINABLE FREE.
Reason: Caixin PMI is licensed via S&P Global Market Intelligence (subscription).
        NBS official PMI is published as press-release numbers on data.stats.gov.cn
        with no free bulk CSV/API for historical daily or monthly series.
Fallback used: FRED BSCICP03CNM665S (OECD business confidence indicator, China,
        manufacturing) -- a proxy only, not the literature's actual PMI series.


## 2. Alignment onto a single daily calendar

All series are reindexed onto copper's trading calendar (NYMEX/COMEX). Monthly FRED-type
series are forward-filled -- this is flagged explicitly as a **look-ahead bias risk**: a
naive forward-fill assumes each monthly value is known immediately at the start of the
month, when in reality PMI-type releases lag their reference month by ~1 business day to
several weeks, and OECD confidence indicators lag considerably more. This exploratory
notebook does not correct for publication lag; a production pipeline must.

In [5]:
close_col_candidates = ["Adj Close", "Close"]


def get_close_series(df, name):
    for c in close_col_candidates:
        if c in df.columns:
            s = df[c].copy()
            s.name = name
            return s
    raise KeyError(f"No Close column found for {name}: {df.columns.tolist()}")


price_frames = {}
for name, df in yf_data.items():
    price_frames[name] = get_close_series(df, name)

# Copper trading calendar is the anchor
calendar_index = price_frames["copper"].dropna().index

prices = pd.DataFrame(index=calendar_index)
for name, s in price_frames.items():
    prices[name] = s.reindex(calendar_index)

# FRED daily series (real yield) reindexed + ffilled (business-day gaps, holidays)
DAILY_FRED = ["real_yield_10y"]
MONTHLY_FRED = ["high_yield_oas", "china_pmi_proxy_oecd_business_confidence"]
# note: BAMLH0A0HYM2 is actually daily in principle, but our pulled window is short; treat
# by its native frequency (ffill for calendar gaps only, not monthly-style)
NATIVE_DAILY_FRED = ["real_yield_10y", "high_yield_oas"]
NATIVE_MONTHLY_FRED = ["china_pmi_proxy_oecd_business_confidence"]

for name in NATIVE_DAILY_FRED:
    if name in fred_data:
        col = fred_data[name].columns[0]
        s = fred_data[name][col].reindex(calendar_index.union(fred_data[name].index)).sort_index()
        s = s.ffill()
        prices[name] = s.reindex(calendar_index)

for name in NATIVE_MONTHLY_FRED:
    if name in fred_data:
        col = fred_data[name].columns[0]
        s = fred_data[name][col].reindex(calendar_index.union(fred_data[name].index)).sort_index()
        s = s.ffill()
        prices[name] = s.reindex(calendar_index)

print("Aligned price panel shape:", prices.shape)
prices.to_csv(RAW_DIR / "aligned_prices_panel.csv")
prices.tail()

Aligned price panel shape: (4181, 12)


,copper,aluminum,gold,silver,crude_oil,dxy,sp500,vix,us10y_yield,real_yield_10y,high_yield_oas,china_pmi_proxy_oecd_business_confidence
Date,,,,,,,,,,,,
2026-08-12,6.5970,3423.00,4408.899902,65.555000,83.269997,100.010002,7748.500000,14.55,4.682,2.42,2.71,97.68838
2026-08-13,6.5925,3391.00,4363.600098,64.873001,81.250000,99.959999,7798.990234,14.63,4.641,2.39,2.71,97.68838
2026-08-14,6.5995,3355.50,4380.399902,64.987999,82.400002,99.669998,7785.759766,14.25,4.696,2.41,2.67,97.68838
2026-08-17,6.6040,3427.25,4417.799805,66.121002,84.500000,99.639999,7745.060059,15.19,4.724,2.44,2.70,97.68838
2026-08-18,6.4505,3343.00,4389.000000,63.290001,84.459999,99.643997,7691.759766,15.84,4.706,2.44,2.70,97.68838


## 2a. Data quality report

In [6]:
dq_rows = []
for col in prices.columns:
    s = prices[col]
    valid = s.dropna()
    dq_rows.append({
        "variable": col,
        "start_date": str(valid.index.min().date()) if len(valid) else None,
        "end_date": str(valid.index.max().date()) if len(valid) else None,
        "n_obs": int(len(valid)),
        "n_expected_calendar_days": int(len(s)),
        "pct_missing_after_alignment": round(100 * (1 - len(valid) / len(s)), 2) if len(s) else None,
    })
dq_report = pd.DataFrame(dq_rows).set_index("variable")
dq_report.to_csv(RESULTS_DIR / "data_quality_report.csv")
print(dq_report)

                                          start_date    end_date  n_obs  \
variable                                                                  
copper                                    2010-01-04  2026-08-18   4181   
aluminum                                  2014-05-06  2026-08-18   3053   
gold                                      2010-01-04  2026-08-18   4180   
silver                                    2010-01-04  2026-08-18   4180   
crude_oil                                 2010-01-04  2026-08-18   4180   
dxy                                       2010-01-04  2026-08-18   4179   
sp500                                     2010-01-04  2026-08-18   4178   
vix                                       2010-01-04  2026-08-18   4178   
us10y_yield                               2010-01-04  2026-08-18   4177   
real_yield_10y                            2010-01-04  2026-08-18   4181   
high_yield_oas                            2023-08-21  2026-08-18    754   
china_pmi_proxy_oecd_busi

## 3. Stationarity: ADF test on levels vs. log returns

Standard result expected: price *levels* are non-stationary (unit root, ADF fails to
reject H0), log *returns* are stationary. This justifies using returns (not levels) for
correlation analysis, and is checked explicitly here rather than assumed.

In [7]:
def adf_report(series, name):
    s = series.dropna()
    if len(s) < 30:
        return {"variable": name, "adf_stat": np.nan, "p_value": np.nan, "n_obs": len(s),
                "stationary_at_5pct": None, "note": "insufficient data"}
    try:
        stat, pval, *_ = adfuller(s, autolag="AIC")
        return {"variable": name, "adf_stat": stat, "p_value": pval, "n_obs": len(s),
                "stationary_at_5pct": bool(pval < 0.05), "note": ""}
    except Exception as e:
        return {"variable": name, "adf_stat": np.nan, "p_value": np.nan, "n_obs": len(s),
                "stationary_at_5pct": None, "note": str(e)}


# Yield/spread-type series (percent, can be zero or negative -- e.g. DFII10 real yield went
# negative 2020-2021) cannot be log-transformed (log of a negative number is undefined). For
# these we use simple first differences (basis-point changes) instead of log returns, which
# is the standard convention for rate/spread series. All price-type series (copper, metals,
# equities, DXY, VIX) use log returns as originally planned.
RATE_LIKE_COLUMNS = {"us10y_yield", "real_yield_10y", "high_yield_oas"}

log_returns = pd.DataFrame(index=prices.index)
for col in prices.columns:
    if col in RATE_LIKE_COLUMNS:
        log_returns[col] = prices[col].diff()  # simple diff, not log-diff -- see note above
    else:
        log_returns[col] = np.log(prices[col]).diff()

adf_levels = [adf_report(prices[c], c) for c in prices.columns]
adf_returns = [adf_report(log_returns[c], c) for c in log_returns.columns]

adf_levels_df = pd.DataFrame(adf_levels).set_index("variable").add_prefix("levels_")
adf_returns_df = pd.DataFrame(adf_returns).set_index("variable").add_prefix("returns_")
adf_combined = adf_levels_df.join(adf_returns_df)
adf_combined.to_csv(RESULTS_DIR / "adf_stationarity_report.csv")
print(adf_combined[["levels_p_value", "levels_stationary_at_5pct",
                     "returns_p_value", "returns_stationary_at_5pct"]])

                                          levels_p_value  \
variable                                                   
copper                                      9.495867e-01   
aluminum                                    7.304827e-01   
gold                                        9.963651e-01   
silver                                      7.149651e-01   
crude_oil                                   9.820786e-02   
dxy                                         3.998457e-01   
sp500                                       9.987599e-01   
vix                                         9.490295e-07   
us10y_yield                                 6.910437e-01   
real_yield_10y                              6.976307e-01   
high_yield_oas                              6.850740e-02   
china_pmi_proxy_oecd_business_confidence    9.026560e-05   

                                          levels_stationary_at_5pct  \
variable                                                              
copper           

## 4. Pearson + Spearman correlation on RETURNS, full heatmap

In [8]:
candidates = [c for c in log_returns.columns if c != "copper"]
ret_clean = log_returns.dropna(subset=["copper"])

corr_rows = []
for cand in candidates:
    pair = ret_clean[["copper", cand]].dropna()
    if len(pair) < 30:
        corr_rows.append({"variable": cand, "n_obs": len(pair),
                           "pearson_r": np.nan, "pearson_p": np.nan,
                           "spearman_r": np.nan, "spearman_p": np.nan})
        continue
    pr, pp = stats.pearsonr(pair["copper"], pair[cand])
    sr, sp = stats.spearmanr(pair["copper"], pair[cand])
    corr_rows.append({"variable": cand, "n_obs": len(pair),
                       "pearson_r": pr, "pearson_p": pp,
                       "spearman_r": sr, "spearman_p": sp})

corr_report = pd.DataFrame(corr_rows).set_index("variable")
corr_report.to_csv(RESULTS_DIR / "return_correlation_report.csv")
print(corr_report)

# Full pairwise heatmap (all variables' returns)
full_corr_matrix = log_returns.corr(method="pearson")
full_corr_matrix.to_csv(RESULTS_DIR / "full_return_correlation_matrix.csv")

plt.figure(figsize=(11, 9))
sns.heatmap(full_corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={"label": "Pearson r (log returns)"})
plt.title("Pairwise Pearson Correlation of Daily Log Returns\n(all candidate variables, 2010-present)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "correlation_heatmap.png", dpi=150)
plt.close()
print("Saved correlation_heatmap.png")

                                          n_obs  pearson_r      pearson_p  \
variable                                                                    
aluminum                                   3047   0.413399  4.197152e-126   
gold                                       4178   0.316352   9.318984e-98   
silver                                     4178   0.438629  4.791879e-196   
crude_oil                                  4176   0.257333   3.799962e-64   
dxy                                        4176  -0.298877   6.369767e-87   
sp500                                      4174   0.304278   3.896223e-90   
vix                                        4174  -0.256083   1.714393e-63   
us10y_yield                                4172   0.124259   7.956233e-16   
real_yield_10y                             4180  -0.005101   7.416103e-01   
high_yield_oas                              753  -0.201298   2.522118e-08   
china_pmi_proxy_oecd_business_confidence   4180   0.010247   5.077830e-01   

Saved correlation_heatmap.png


## 5. Rolling 252-trading-day correlation between copper returns and each candidate

In [9]:
ROLL_WINDOW = 252
rolling_corrs = {}
for cand in candidates:
    pair = ret_clean[["copper", cand]]
    rc = pair["copper"].rolling(ROLL_WINDOW).corr(pair[cand])
    rolling_corrs[cand] = rc
    if rc.dropna().empty:
        continue
    plt.figure(figsize=(12, 4))
    plt.plot(rc.index, rc.values, color="steelblue", linewidth=1)
    plt.axhline(0, color="black", linewidth=0.8, linestyle="--")
    plt.title(f"252-day Rolling Correlation: Copper Returns vs. {cand} Returns")
    plt.ylabel("Pearson r")
    plt.ylim(-1, 1)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"rolling_corr_{cand}.png", dpi=130)
    plt.close()

rolling_df = pd.DataFrame(rolling_corrs)
rolling_df.to_csv(RESULTS_DIR / "rolling_correlations.csv")
print("Saved rolling correlation plots for:", list(rolling_corrs.keys()))

# Specific check: does China-PMI-proxy correlation break down near zero around 2022?
if "china_pmi_proxy_oecd_business_confidence" in rolling_df.columns:
    pmi_roll = rolling_df["china_pmi_proxy_oecd_business_confidence"].dropna()
    around_2022 = pmi_roll[(pmi_roll.index >= "2021-06-01") & (pmi_roll.index <= "2023-06-01")]
    print("\nChina-PMI-proxy rolling corr, mid-2021 to mid-2023 (checking claimed 2022 breakdown):")
    print(around_2022.describe())

Saved rolling correlation plots for: ['aluminum', 'gold', 'silver', 'crude_oil', 'dxy', 'sp500', 'vix', 'us10y_yield', 'real_yield_10y', 'high_yield_oas', 'china_pmi_proxy_oecd_business_confidence']

China-PMI-proxy rolling corr, mid-2021 to mid-2023 (checking claimed 2022 breakdown):
count    505.000000
mean      -0.045039
std        0.064057
min       -0.139298
25%       -0.096717
50%       -0.062391
75%       -0.016093
max        0.074646
Name: china_pmi_proxy_oecd_business_confidence, dtype: float64


## 6. Lead-lag cross-correlation function, lags -10 to +10 trading days

Positive lag k means candidate's return at t-k predicts (leads) copper's return at t --
i.e. we correlate `copper_return[t]` with `candidate_return[t-k]`. A peak at positive k
means the candidate genuinely LEADS copper (useful predictive feature); a peak at k=0 means
purely contemporaneous relationship (not useful for prediction without same-day data);
negative-k peak means the candidate LAGS copper (copper leads it, not useful as predictor).

In [10]:
LAGS = range(-10, 11)


def ccf_at_lags(copper_ret, cand_ret, lags):
    out = {}
    for k in lags:
        if k >= 0:
            x = copper_ret
            y = cand_ret.shift(k)
        else:
            x = copper_ret
            y = cand_ret.shift(k)  # negative shift = shift backward (future values)
        pair = pd.concat([x, y], axis=1).dropna()
        if len(pair) < 30:
            out[k] = np.nan
            continue
        r, _ = stats.pearsonr(pair.iloc[:, 0], pair.iloc[:, 1])
        out[k] = r
    return out


ccf_results = {}
for cand in candidates:
    copper_ret = ret_clean["copper"]
    cand_ret = ret_clean[cand]
    ccf_results[cand] = ccf_at_lags(copper_ret, cand_ret, LAGS)

    lag_vals = list(LAGS)
    corr_vals = [ccf_results[cand][k] for k in lag_vals]
    plt.figure(figsize=(8, 4))
    plt.bar(lag_vals, corr_vals, color=["crimson" if k < 0 else ("gray" if k == 0 else "steelblue") for k in lag_vals])
    plt.axhline(0, color="black", linewidth=0.8)
    plt.title(f"Lead-Lag CCF: Copper Returns vs. {cand} Returns (lag in trading days)\n"
              f"(negative=candidate lags copper, positive=candidate leads copper)")
    plt.xlabel("Lag (days); candidate return shifted by k")
    plt.ylabel("Pearson r")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"ccf_{cand}.png", dpi=130)
    plt.close()

ccf_df = pd.DataFrame(ccf_results).T
ccf_df.columns = [f"lag_{k}" for k in LAGS]
ccf_df.to_csv(RESULTS_DIR / "lead_lag_ccf.csv")
print("Saved CCF plots for:", list(ccf_results.keys()))
print(ccf_df)

Saved CCF plots for: ['aluminum', 'gold', 'silver', 'crude_oil', 'dxy', 'sp500', 'vix', 'us10y_yield', 'real_yield_10y', 'high_yield_oas', 'china_pmi_proxy_oecd_business_confidence']
                                           lag_-10    lag_-9    lag_-8  \
aluminum                                 -0.000706  0.000468 -0.032480   
gold                                      0.027246 -0.002095 -0.016574   
silver                                    0.015094  0.004915 -0.020109   
crude_oil                                -0.034112 -0.028233  0.008970   
dxy                                      -0.002906  0.017680 -0.019756   
sp500                                     0.020848  0.003778 -0.024651   
vix                                      -0.014999  0.010953  0.004348   
us10y_yield                              -0.037915  0.006000 -0.012330   
real_yield_10y                           -0.030907 -0.006226 -0.016131   
high_yield_oas                           -0.006677  0.010261  0.055439   
chi

## 7. Granger causality: does candidate Granger-cause copper returns? lags 1, 5, 22

In [11]:
GRANGER_LAGS = [1, 5, 22]
granger_results = {}
for cand in candidates:
    pair = ret_clean[["copper", cand]].dropna()
    row = {}
    if len(pair) < 300:
        for L in GRANGER_LAGS:
            row[f"granger_p_lag{L}"] = np.nan
        row["note"] = "insufficient data"
        granger_results[cand] = row
        continue
    # grangercausalitytests expects [effect, cause] column order, tests cause -> effect
    gc_input = pair[["copper", cand]].values
    try:
        gc_out = grangercausalitytests(gc_input, maxlag=max(GRANGER_LAGS), verbose=False)
        for L in GRANGER_LAGS:
            pval = gc_out[L][0]["ssr_ftest"][1]
            row[f"granger_p_lag{L}"] = pval
        row["note"] = ""
    except Exception as e:
        for L in GRANGER_LAGS:
            row[f"granger_p_lag{L}"] = np.nan
        row["note"] = str(e)
    granger_results[cand] = row

granger_df = pd.DataFrame(granger_results).T
granger_df.to_csv(RESULTS_DIR / "granger_causality_report.csv")
print(granger_df)

                                         granger_p_lag1 granger_p_lag5  \
aluminum                                        0.86243       0.999875   
gold                                            0.57518       0.349544   
silver                                         0.075888       0.490223   
crude_oil                                      0.159734        0.62504   
dxy                                            0.000014       0.001358   
sp500                                          0.000014       0.000006   
vix                                            0.000067       0.000041   
us10y_yield                                    0.516856       0.087452   
real_yield_10y                                 0.002715       0.002179   
high_yield_oas                                 0.014041       0.132047   
china_pmi_proxy_oecd_business_confidence       0.514994       0.531499   

                                         granger_p_lag22 note  
aluminum                                       

## 8. Cointegration (Engle-Granger) between copper PRICE LEVEL and each candidate's level

Cointegration is a *levels* concept, distinct from return correlation. Specifically
relevant to DXY, which the literature review flags as the most robust cross-asset
relationship it found.

In [12]:
coint_results = {}
for cand in candidates:
    pair = prices[["copper", cand]].dropna()
    if len(pair) < 100:
        coint_results[cand] = {"coint_stat": np.nan, "p_value": np.nan, "n_obs": len(pair)}
        continue
    try:
        stat, pval, crit = coint(pair["copper"], pair[cand])
        coint_results[cand] = {"coint_stat": stat, "p_value": pval, "n_obs": len(pair)}
    except Exception as e:
        coint_results[cand] = {"coint_stat": np.nan, "p_value": np.nan, "n_obs": len(pair), "note": str(e)}

coint_df = pd.DataFrame(coint_results).T
coint_df.to_csv(RESULTS_DIR / "cointegration_report.csv")
print(coint_df)

                                          coint_stat   p_value   n_obs
aluminum                                   -3.269072  0.059116  3053.0
gold                                       -2.880584  0.141386  4180.0
silver                                     -2.720590  0.192261  4180.0
crude_oil                                  -0.685627  0.948167  4180.0
dxy                                        -0.340839  0.973916  4179.0
sp500                                      -1.891653  0.584234  4178.0
vix                                        -0.378367  0.971947  4178.0
us10y_yield                                -1.544234  0.743843  4177.0
real_yield_10y                             -1.214549  0.854050  4181.0
high_yield_oas                             -1.588539  0.725703   754.0
china_pmi_proxy_oecd_business_confidence   -0.565440  0.959333  4181.0


## 9. Verdict table

One row per candidate. Combines: contemporaneous return correlation + p-value, best CCF
lag + correlation there, Granger p-values at lags 1/5/22, cointegration p-value, and a
plain-English verdict cross-checked against what the literature review docs claimed.

In [13]:
LITERATURE_CLAIMS = {
    "dxy": "Negative correlation with copper, typically -0.3 to -0.6 (copper_fundamentals.md 3.2); "
           "'most reliable macro feature' cross-asset relationship (literature_review 2.6).",
    "gold": "'Great influence on copper price' / high correlation (Chen et al. 2023, cited in literature_review 2.6).",
    "silver": "'High influence' on copper (Chen et al. 2023, cited in literature_review 2.6).",
    "crude_oil": "Short-term positive correlation via shared macro drivers (Ling et al. 2025, literature_review 2.6).",
    "sp500": "Not a headline claim in the docs; implicitly a risk-sentiment proxy alongside VIX.",
    "vix": "Negative correlation; VIX spikes coincide with copper sell-offs (copper_fundamentals.md 3.4).",
    "aluminum": "Not specifically quantified in either doc; grouped as 'other base metals' cross-market co-movement.",
    "us10y_yield": "Not a headline claim; grouped under 'interest rates' as a macro driver (literature_review intro).",
    "real_yield_10y": "Not directly discussed in either doc (nominal yields only mentioned); Block C candidate per CuBench plan.",
    "high_yield_oas": "Not discussed in either doc; Block C 'credit spreads' candidate per CuBench plan.",
    "china_pmi_proxy_oecd_business_confidence": "China PMI: 'positive correlation, particularly at 1-3 month horizons' "
        "(copper_fundamentals.md 3.3); literature_review flags 'lagged effect observed' (Tang et al. 2026) and this "
        "project's own econometrics review flagged a reported 2022 breakdown to near-zero.",
}


def classify_verdict(row):
    p = row.get("pearson_p", np.nan)
    r = row.get("pearson_r", np.nan)
    best_lag = row.get("best_ccf_lag", np.nan)
    best_lag_r = row.get("best_ccf_corr", np.nan)
    g1, g5, g22 = row.get("granger_p_lag1"), row.get("granger_p_lag5"), row.get("granger_p_lag22")
    coint_p = row.get("coint_p_value", np.nan)
    rolling_std = row.get("rolling_corr_std", np.nan)
    rolling_min = row.get("rolling_corr_min", np.nan)
    rolling_max = row.get("rolling_corr_max", np.nan)

    if pd.isna(p) or pd.isna(r):
        return "could not verify (data quality issue)"

    contemp_sig = p < 0.05
    any_granger_sig = any(v is not None and not pd.isna(v) and v < 0.05 for v in [g1, g5, g22])
    leads = (not pd.isna(best_lag)) and best_lag > 0 and abs(best_lag_r) > abs(r) * 1.05

    # Regime instability: does the 252-day rolling correlation cross zero (sign flip) by a
    # non-trivial margin at some point in the 14-year sample, or have very high dispersion?
    sign_flip = (not pd.isna(rolling_min)) and (not pd.isna(rolling_max)) and \
        (rolling_min < -0.05) and (rolling_max > 0.05)
    high_dispersion = (not pd.isna(rolling_std)) and rolling_std > 0.20
    unstable = sign_flip or high_dispersion

    if not contemp_sig and abs(r) < 0.08:
        return "weak/marginal, likely not decision-useful"
    if unstable and contemp_sig:
        return (f"correlated on average but regime-unstable (252d rolling corr ranges "
                 f"{rolling_min:.2f} to {rolling_max:.2f}, {'crosses zero' if sign_flip else 'high dispersion'} "
                 f"across the 2010-present sample -- not a stable structural relationship)")
    if leads and any_granger_sig:
        return "genuinely correlated with a genuine lead -- decision-useful as predictive feature"
    if contemp_sig and not leads:
        return "contemporaneous only, no genuine lead at daily frequency -- not a useful predictive feature as-is"
    if contemp_sig:
        return "genuinely correlated, broadly consistent across the sample"
    return "weak/marginal, likely not decision-useful"


verdict_rows = []
for cand in candidates:
    r = corr_report.loc[cand, "pearson_r"] if cand in corr_report.index else np.nan
    p = corr_report.loc[cand, "pearson_p"] if cand in corr_report.index else np.nan
    sr = corr_report.loc[cand, "spearman_r"] if cand in corr_report.index else np.nan

    ccf_row = ccf_df.loc[cand] if cand in ccf_df.index else None
    if ccf_row is not None and ccf_row.notna().any():
        abs_ccf = ccf_row.abs()
        best_lag_col = abs_ccf.idxmax()
        best_lag = int(best_lag_col.replace("lag_", ""))
        best_lag_corr = ccf_row[best_lag_col]
    else:
        best_lag, best_lag_corr = np.nan, np.nan

    g_row = granger_df.loc[cand] if cand in granger_df.index else {}
    g1 = g_row.get("granger_p_lag1", np.nan)
    g5 = g_row.get("granger_p_lag5", np.nan)
    g22 = g_row.get("granger_p_lag22", np.nan)

    coint_p = coint_df.loc[cand, "p_value"] if cand in coint_df.index else np.nan

    rc = rolling_df[cand].dropna() if cand in rolling_df.columns else pd.Series(dtype=float)
    rolling_std = rc.std() if len(rc) else np.nan
    rolling_min = rc.min() if len(rc) else np.nan
    rolling_max = rc.max() if len(rc) else np.nan

    row = {
        "variable": cand,
        "pearson_r": r, "pearson_p": p, "spearman_r": sr,
        "best_ccf_lag": best_lag, "best_ccf_corr": best_lag_corr,
        "granger_p_lag1": g1, "granger_p_lag5": g5, "granger_p_lag22": g22,
        "coint_p_value": coint_p,
        "rolling_corr_std": rolling_std, "rolling_corr_min": rolling_min, "rolling_corr_max": rolling_max,
    }
    row["verdict"] = classify_verdict(row)
    row["literature_claim"] = LITERATURE_CLAIMS.get(cand, "No specific claim found in the reviewed docs.")
    verdict_rows.append(row)

verdict_df = pd.DataFrame(verdict_rows).set_index("variable")
verdict_df.to_csv(RESULTS_DIR / "summary_verdict.csv")

with open(RESULTS_DIR / "summary_verdict.json", "w") as f:
    json.dump({
        "generated": datetime.today().isoformat(),
        "start_date": START_DATE,
        "end_date": END_DATE,
        "data_quality": dq_report.reset_index().to_dict(orient="records"),
        "adf_stationarity": adf_combined.reset_index().to_dict(orient="records"),
        "return_correlation": corr_report.reset_index().to_dict(orient="records"),
        "lead_lag_ccf": ccf_df.reset_index().rename(columns={"index": "variable"}).to_dict(orient="records"),
        "granger_causality": granger_df.reset_index().rename(columns={"index": "variable"}).to_dict(orient="records"),
        "cointegration": coint_df.reset_index().rename(columns={"index": "variable"}).to_dict(orient="records"),
        "verdict": verdict_df.reset_index().to_dict(orient="records"),
        "china_pmi_genuine_series": "NOT_OBTAINABLE_FREE",
        "china_pmi_fallback_used": "FRED BSCICP03CNM665S (OECD business confidence indicator, China manufacturing, monthly proxy)",
    }, f, indent=2, default=str)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(verdict_df)

                                          pearson_r      pearson_p  spearman_r  best_ccf_lag  best_ccf_corr  granger_p_lag1  granger_p_lag5  granger_p_lag22  coint_p_value  rolling_corr_std  \
variable                                                                                                                                                                                        
aluminum                                   0.413399  4.197152e-126    0.429573             0       0.413399        0.862430        0.999875         0.159940       0.059116          0.224046   
gold                                       0.316352   9.318984e-98    0.305480             0       0.316352        0.575180        0.349544         0.324301       0.141386          0.168378   
silver                                     0.438629  4.791879e-196    0.434127             0       0.438629        0.075888        0.490223         0.318946       0.192261          0.138927   
crude_oil                          

## 10. Cross-check against literature claims -- explicit disconfirmation flags

In [14]:
for cand, row in verdict_df.iterrows():
    print(f"\n=== {cand} ===")
    print(f"Literature claim: {row['literature_claim']}")
    print(f"Empirical verdict: {row['verdict']}")
    print(f"  pearson r={row['pearson_r']:.3f} (p={row['pearson_p']:.4f}), "
          f"best CCF lag={row['best_ccf_lag']} (r={row['best_ccf_corr']:.3f} if not nan), "
          f"coint p={row['coint_p_value']:.4f}")

print("\nDone. All outputs written to:", RESULTS_DIR)


=== aluminum ===
Literature claim: Not specifically quantified in either doc; grouped as 'other base metals' cross-market co-movement.
Empirical verdict: correlated on average but regime-unstable (252d rolling corr ranges -0.16 to 0.66, crosses zero across the 2010-present sample -- not a stable structural relationship)
  pearson r=0.413 (p=0.0000), best CCF lag=0 (r=0.413 if not nan), coint p=0.0591

=== gold ===
Literature claim: 'Great influence on copper price' / high correlation (Chen et al. 2023, cited in literature_review 2.6).
Empirical verdict: correlated on average but regime-unstable (252d rolling corr ranges -0.11 to 0.63, crosses zero across the 2010-present sample -- not a stable structural relationship)
  pearson r=0.316 (p=0.0000), best CCF lag=0 (r=0.316 if not nan), coint p=0.1414

=== silver ===
Literature claim: 'High influence' on copper (Chen et al. 2023, cited in literature_review 2.6).
Empirical verdict: contemporaneous only, no genuine lead at daily frequency 